In [3]:
#prevents warning froms flooding the console 
import warnings
warnings.filterwarnings("ignore")

# After all your imports, set it again
def custom_warn(*args, **kwargs):
    pass

warnings.showwarning = custom_warn  # Completely override the warning display function

In [4]:
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
import joblib

#loading model from previous phase
model = joblib.load("../4_training_model/stroke_classifer_model.pkl")
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

feature_columns = [
    'RIGHT_WRIST_x', 'RIGHT_WRIST_y', 'RIGHT_WRIST_z',
    'RIGHT_ELBOW_x', 'RIGHT_ELBOW_y', 'RIGHT_ELBOW_z',
    'RIGHT_SHOULDER_x', 'RIGHT_SHOULDER_y', 'RIGHT_SHOULDER_z',
    'LEFT_SHOULDER_x', 'LEFT_SHOULDER_y', 'LEFT_SHOULDER_z',
    'LEFT_HIP_x', 'LEFT_HIP_y', 'LEFT_HIP_z',
    'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_HIP_z',
    'RIGHT_ANKLE_x', 'RIGHT_ANKLE_y', 'RIGHT_ANKLE_z',
    'LEFT_ANKLE_x', 'LEFT_ANKLE_y', 'LEFT_ANKLE_z',
]

# Function for live webcam detection strokes - very similar to video

In [5]:
def detect_strokes_live(output_csv_path):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print(f"Can't open webcam")
        return
    
    print(f"Opened webcam successfully")
    
    #video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Webcam: {frame_width} x {frame_height}, FPS: {fps}")
    
    detections = []
    
    with mp_pose.Pose() as pose:
        frame_count = 0
        
        while cap.isOpened():
            success, frame = cap.read()
            
            if not success:
                break
            
            #------------Phase 2: Looks at key landmarks ------------#
            
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(frame_rgb)
            
            if results.pose_landmarks:
                landmark_dict = {}
                
                for landmark in mp_pose.PoseLandmark:
                    lm = results.pose_landmarks.landmark[landmark]
                    landmark_dict[f"{landmark.name}_x"] = lm.x
                    landmark_dict[f"{landmark.name}_y"] = lm.y
                    landmark_dict[f"{landmark.name}_z"] = lm.z
                
                #------------Phase 3/4: Using model to predict strokes----------#
                
                #gets only the values from the featured cols defined previously
                features = [landmark_dict.get(col, 0) for col in feature_columns]
                
                #converts input into 2d array (1,24) for the model to predict one
                X = np.array(features).reshape(1, -1)
                
                #returns the one prediction (string) about the stroke
                stroke_pred = model.predict(X)[0]      
                
                #returns the highest prob of the 4 strokes
                confidence = model.predict_proba(X).max() 
                
                stroke_colors = {
                    'Forehand': (50, 220, 50),
                    'Backhand': (50, 180, 255),
                    'Serve': (255, 160, 50),
                    'Neutral': (180, 180, 180)
                }
                
                #adds stroke and confidence in corner 
                color = stroke_colors.get(stroke_pred, (255,255,255))
                text = f"{stroke_pred} ({confidence:.2f})"
                cv2.putText(frame, text, (30,50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)
                cv2.imshow("SpinTracker Live", frame)
            
                #writes to csv
                timestamp = frame_count/fps
                detections.append({
                    'frame': frame_count,
                    'timestamp': timestamp,
                    'stroke': stroke_pred,
                    'confidence': confidence
                })
           # else:
            #    print(f"No landmarks detected at {(frame_count/fps):.2f}")

            frame_count += 1            
            if frame_count % 100 == 0:
                print(f"Processed {frame_count} frames")
                
            key = cv2.waitKey(1) & 0xFF    
            if key == ord('q'):
                print("Detection session ended")
                break
    
    df = pd.DataFrame(detections)
    df.to_csv(output_csv_path, index= False)
    print(f"Successfully saved to {output_csv_path}")
    cap.release()

# Live webcam Testing

In [7]:
detect_strokes_live("live_session.csv")

Opened webcam successfully
Webcam: 640 x 480, FPS: 30.0
Processed 100 frames
Processed 200 frames
Processed 300 frames
Processed 400 frames
Processed 500 frames
Processed 600 frames
Detection session ended
Successfully saved to live_session.csv
